# AGFN denovo: molecule + docking visualization

Three views of what the model is generating:

1. **Trajectory step grids** — partial molecules at each GFN action step (RDKit 2D).
2. **Top-K across training** — best generated molecules ranked by reward or Uni-Dock affinity, from `<log_dir>/top_k_mols.pt` produced by the `TopKTracker` hook in `samp_iter_finetune.py`.
3. **3D docked pose** — a fresh batch redocked into a persistent directory, rendered with py3Dmol.

**Runs from anywhere in the repo** (this notebook lives in `./notebooks/`). All paths in `denovo.yml` are relative; cell 2 walks up to the repo root and `chdir`s there for you. Requires `py3Dmol` (in `requirements_jn.txt`).

In [ ]:
# ----- User parameters (edit these) -----
CONFIG_PATH      = "./src/config/denovo.yml"
SAVED_MODEL_PATH = None     # None -> use hps.saved_model_path from YAML
TARGET_NAME      = None     # None -> use hps.target_name from YAML
N_SAMPLES        = 8        # trajectories to draw
N_DOCK           = 4        # subset to dock
TOP_K_SHOW       = 20       # rows of top-K to render (max 100)
TOP_K_RANK_BY    = "reward" # "reward" or "affinity"
PERSIST_DIR      = "./AGFN_logs/visualize_out"
DEVICE_ID        = 0
SEED             = 0

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import torch

# The only boilerplate: make `import nbtools` resolve, then bootstrap the repo (chdir to root +
# put src/ on sys.path). Importing nbtools loads py3Dmol first, so rdkit.Chem.Draw stays safe.
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "nbtools").is_dir():
        sys.path.insert(0, str(_c)); break
    if (_c / "notebooks" / "nbtools").is_dir():
        sys.path.insert(0, str(_c / "notebooks")); break
import nbtools
from nbtools import sampling, render2d, render3d, docking, hall_of_fame
REPO_ROOT = nbtools.setup_repo()
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
torch.manual_seed(SEED); np.random.seed(SEED)
print("repo root:", REPO_ROOT)

In [ ]:
# Load + patch hps for the QedxSaxDock denovo task, then set up output dirs.
hps, conditional_range_dict, cond_prop_var = nbtools.load_denovo_hps(
    CONFIG_PATH, saved_model_path=SAVED_MODEL_PATH, target_name=TARGET_NAME)
gfn_samples_path = f"{hps.gfn_samples_path}/GFN_gen_samples_{hps.target_name}/"
os.makedirs(gfn_samples_path, exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)
print("target_name:     ", hps.target_name)
print("saved_model_path:", hps.saved_model_path)
print("log_dir:         ", hps.log_dir)
print("gfn_samples_path:", gfn_samples_path)

In [ ]:
# Build DockingFineTuner and load the checkpoint -- reuses the exact constructor used by training,
# so model architecture and checkpoint loading match (see nbtools.sampling.build_finetuner).
finetuner = sampling.build_finetuner(
    hps, conditional_range_dict, cond_prop_var, hps.saved_model_path,
    rank=DEVICE_ID, world_size=1, gfn_samples_path=gfn_samples_path)
print("device:", finetuner.device)

In [ ]:
# Sample N_SAMPLES trajectories (free generation, no seed); keep the valid ones.
valid = sampling.sample_trajectories(finetuner, N_SAMPLES)
print(f"requested {N_SAMPLES} trajectories, {len(valid)} valid")

In [ ]:
# Trajectory step grids for the first few trajectories. traj[t][0] is already the pre-action Graph
# state -- no need to replay env.step to reconstruct intermediates.
from IPython.display import display
for idx, t in enumerate(valid[:3]):
    img, nsteps = render2d.trajectory_step_grid(t["traj"], finetuner.ctx)
    if img is None:
        print(f"trajectory {idx}: no renderable intermediate graphs")
        continue
    print(f"trajectory {idx}  steps={nsteps}")
    display(img)

In [ ]:
# Top-K best generated molecules across the whole training run, from <log_dir>/top_k_mols.pt
# (written by the TopKTracker hook in samp_iter_finetune.py). Skips gracefully if it isn't there.
from IPython.display import display
img = hall_of_fame.topk_pt_grid(hps.log_dir, rank_by=TOP_K_RANK_BY, n_show=TOP_K_SHOW)
if img is not None:
    display(img)

In [ ]:
# Extract final SMILES from the live sample for re-docking.
from rdkit import Chem
smiles, final_mols = [], []
for t in valid[:N_DOCK]:
    try:
        m = finetuner.ctx.graph_to_mol(t["traj"][-1][0])
        s = Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        s, m = None, None
    if s:
        smiles.append(s); final_mols.append(m)
for i, s in enumerate(smiles):
    print(f"  [{i}] {s}")
print(f"will dock {len(smiles)} molecules")

In [ ]:
# Dock the batch with Uni-Dock (in-process). Pose .sdf files persist under PERSIST_DIR so the 3D
# viewer below can read them -- same ETKDG + UniDock backend as training (nbtools.docking).
grid = dict(hps.target_grid[hps.target_name])
receptor = grid["receptor"]
center = (grid["center_x"], grid["center_y"], grid["center_z"])
size   = (grid["size_x"],   grid["size_y"],   grid["size_z"])
dock_dir = Path(PERSIST_DIR) / f"{hps.target_name}_unidock"

result = docking.dock_batch(smiles, receptor, center, size, dock_dir,
                            search_mode=hps.get("unidock_search_mode", "fast"), seed=SEED)
smiles_out, affinities, rewards, pose_paths = (
    result["smiles"], result["affinities"], result["rewards"], result["pose_paths"])
print("results")
print(f"  poses: {result['save_dir']}")
for i, (s, a, r) in enumerate(zip(smiles_out, affinities, rewards)):
    print(f"  [{i}] aff={a:+.2f}  reward={r:+.3f}  smi={s}")

In [ ]:
# 3D pose viewer. Receptor rainbow cartoon + ligand green-carbon sticks.
for i, pose in enumerate(pose_paths):
    if pose is None:
        print(f"[{i}] no pose file (docking likely failed)")
        continue
    print(f"\n[{i}]  aff={affinities[i]:+.2f}  smi={smiles_out[i]}")
    render3d.view_pose(receptor, pose).show()

In [ ]:
# Optional summary save (JSON, no binary objects). Trajectories regenerate from the same
# checkpoint + seed, so we don't persist them.
summary = {
    "target_name":      hps.target_name,
    "saved_model_path": hps.saved_model_path,
    "smiles":           list(smiles_out),
    "affinities":       [float(a) for a in affinities],
    "rewards":          [float(r) for r in rewards],
    "pose_paths":       pose_paths,
    "receptor":         receptor,
}
out_path = os.path.join(PERSIST_DIR, f"{hps.target_name}_visualize_results.json")
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)
print("wrote", out_path)